## Modelo Final y Empaquetado

Objetivo:
- Reconstruir el pipeline con el mejor modelo identificado en MLflow.
- Entrenar el modelo final sobre los datos procesados.
- Evaluarlo en un conjunto de prueba.
- Guardar el pipeline entrenado en disco para que pueda ser usado por la API y el tablero (sin rehacer todo el notebook).

Modelo seleccionado:
- GradientBoostingRegressor
- learning_rate = 0.1
- n_estimators = 200

Criterios:
 - Mejor R² (~0.17) entre los modelos evaluados.
 - RMSE y MAE competitivos y estables.
 - Buen balance entre capacidad de modelar relaciones no lineales y riesgo de sobreajuste.

### Librerías

In [1]:
import os
import pandas as pd
from math import sqrt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import GradientBoostingRegressor
import joblib


### Carga del dataset procesado

In [2]:
# Ruta al dataset procesado (ajustar si es necesario)
ruta_dataset = "../data/processed/viajeros_2023_gasto_cop.csv"

df = pd.read_csv(ruta_dataset)
print("Dataset cargado:", df.shape)
df.head()


Dataset cargado: (6440, 135)


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_1140\1526530193.py:4: DtypeWarning: Columns (22,32,34,35,36,37,57,61,82,83,89,91,93,94,99,104,112,115,116,117,118,119,120,123,124,125) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_dataset)


,Obs_ID,P_101,P_102,P_102A,P_102_B,P_103,P_103A,P_104,P_105,P_105A,...,P_501_K,P_502_A,P_502_B,P_502_C,P_502_D,P_502_E,P_502_F,P_502_G,P_502_H,Gasto_Total_COP
0,1,Si,Nacional,CASANARE,NaN,a. Vacaciones/recreación/Ocio,NaN,a. Hombre,a. Pesos,3000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3000000.0
1,18,Si,Internacional,NaN,Guatemala,b. Visita a familiares y amigos,NaN,b. Mujer,b. Dolares,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2000000.0
2,25,Si,Internacional,NaN,Guatemala,a. Vacaciones/recreación/Ocio,NaN,b. Mujer,a. Pesos,3000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3000000.0
3,32,Si,Internacional,NaN,Estados Unidos,b. Visita a familiares y amigos,NaN,b. Mujer,a. Pesos,1500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500000.0
4,39,Si,Internacional,NaN,Panamá,a. Vacaciones/recreación/Ocio,NaN,a. Hombre,a. Pesos,8000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8000000.0


### Definición de variables

In [3]:
# Nombre de la variable objetivo
TARGET = "Gasto_Total_COP"

# Variables categóricas usadas en el modelo
features_cat = [
    "P_102",   # País / región de origen (ejemplo)
    "P_103",   # Motivo del viaje
    "P_107"    # Tipo de alojamiento
]

# Variables numéricas usadas en el modelo
features_num = [
    "P_106A"   # Número de noches de alojamiento
]

# Subconjunto con las columnas necesarias
columnas_modelo = features_cat + features_num + [TARGET]

df_model = df[columnas_modelo].dropna()
print("Dataset para modelado:", df_model.shape)

X = df_model[features_cat + features_num]
y = df_model[TARGET]


Dataset para modelado: (6130, 5)


### División train / test

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (4904, 4) Test: (1226, 4)


### Preprocesador

In [6]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), features_cat),
        ("num", "passthrough", features_num)
    ]
)


### Definir modelo ganador y pipeline final

In [7]:
# Mejor configuración encontrada en MLflow para Gradient Boosting
mejor_modelo = GradientBoostingRegressor(
    learning_rate=0.1,
    n_estimators=200,
    random_state=42
)

pipeline_final = Pipeline(steps=[
    ("pre", preprocessor),
    ("model", mejor_modelo)
])

pipeline_final


Pipeline(steps=[('pre',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['P_102', 'P_103', 'P_107']),
                                                 ('num', 'passthrough',
                                                  ['P_106A'])])),
                ('model',
                 GradientBoostingRegressor(n_estimators=200, random_state=42))])

### Entrenamiento y evaluación del modelo final

In [9]:
# Entrenar
pipeline_final.fit(X_train, y_train)

# Predicciones en test
y_pred = pipeline_final.predict(X_test)

# Métricas
mse = mean_squared_error(y_test, y_pred)
rmse = sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Métricas del modelo final (Gradient Boosting)")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE : {mae:,.2f}")
print(f"R²  : {r2:.3f}")


Métricas del modelo final (Gradient Boosting)
RMSE: 8,065,023.96
MAE : 2,289,917.15
R²  : 0.117


### Guardar el modelo empaquetado

In [10]:
# Carpeta donde dejaremos el modelo para despliegue
os.makedirs("../models", exist_ok=True)

ruta_modelo = "../models/modelo_gasto_turistico_gb.pkl"
joblib.dump(pipeline_final, ruta_modelo)

print(f"Modelo final guardado en: {ruta_modelo}")


Modelo final guardado en: ../models/modelo_gasto_turistico_gb.pkl


### Comentarios

In [ ]:
"""
Guía de uso del modelo empaquetado
----------------------------------

El archivo ../models/modelo_gasto_turistico_gb.pkl contiene un
Pipeline de scikit-learn con:

    - Preprocesamiento:
        * OneHotEncoder sobre:
            - P_102  (Origen del visitante)
            - P_103  (Motivo del viaje)
            - P_107  (Tipo de alojamiento)
        * 'passthrough' para:
            - P_106A (Noches de alojamiento)

    - Modelo:
        * GradientBoostingRegressor(learning_rate=0.1,
                                   n_estimators=200,
                                   random_state=42)

Para usarlo en una API o en el tablero:

    import joblib
    import pandas as pd

    # 1. Cargar el modelo
    modelo = joblib.load("../models/modelo_gasto_turistico_gb.pkl")

    # 2. Construir un DataFrame con las columnas de entrada EXACTAS:
    #    ['P_102', 'P_103', 'P_107', 'P_106A']

    nuevo_viajero = pd.DataFrame([{
        "P_102":  "Internacional",
        "P_103":  "a. Vacaciones/recreación/Ocio",
        "P_107":  "a. Hotel",
        "P_106A": 5  # noches de alojamiento
    }])

    # 3. Obtener la predicción de gasto total en COP
    prediccion = modelo.predict(nuevo_viajero)[0]

    print(f"Gasto total estimado: {prediccion:,.0f} COP")

Cualquier servicio (FastAPI, Flask, etc.) solo necesita:
    - Cargar este .pkl una vez al iniciar.
    - Recibir estos 4 campos.
    - Armar el DataFrame y llamar modelo.predict().
"""
